# Working with alternative Sources + Sinks in Apache Spark

## Overview of Sinks in Spark
In Apache Spark, a "sink" refers to the destination for data that is being processed. Sinks can be files, databases, or even real-time dashboards. When working with batch data (DataFrames) and streaming data (Spark Streaming), sinks are essential for storing or outputting the final processed results.

## Sinks in Batch Processing
In batch processing, DataFrames are used to perform operations on static data. After processing, the results often need to be stored persistently. Spark supports various output formats and systems as sinks for DataFrames, including:

- **File systems** (e.g., HDFS, S3)
- **Database systems** (e.g., JDBC databases like PostgreSQL)
- **Big Data systems** (e.g., HBase, Cassandra)

## Integrating JDBC with Apache Spark for PostgreSQL

### Introduction to JDBC in Spark
JDBC (Java Database Connectivity) is an API for the Java programming language that defines how a client may access a database. It provides methods for querying and updating data in a database. Apache Spark supports JDBC to allow data read and write operations directly from/to relational databases like PostgreSQL.

### Configuring Spark with PostgreSQL JDBC
To connect Spark with a PostgreSQL database, you need the PostgreSQL JDBC driver. This driver is essential because it enables Spark to interact with the database. Here's how you set up Spark to use the PostgreSQL JDBC driver:

#### Step 1: Include the PostgreSQL JDBC Driver
You need to download the PostgreSQL JDBC driver and make it accessible to Spark. This involves placing the JDBC jar file in a known location and referencing it in your Spark configuration.

In [1]:
import os

# Path to the PostgreSQL JDBC jar file
rel_path = os.path.relpath('./libs/postgresql-42.7.3.jar')

We won't need kafka right now but we'll initizalize it for later use

In [2]:
# Kafka configuration parameters for Confluent Cloud
# Kafka configuration parameters for Confluent Cloud
kafka_params = {
      "kafka.bootstrap.servers": "46.225.20.89:9092",
      "subscribe": "commerce-fhtw",
      "kafka.security.protocol": "PLAINTEXT",
      "startingOffsets": "latest",
}

### Configure your spark session to include custom jar files

In Apache Spark, external libraries, such as JDBC drivers or connectors for other data sources like Apache Kafka, can be included in your Spark jobs by configuring the Spark session

In [5]:
import os
from pyspark.sql import SparkSession

# Create a Spark session and configure it to use the JDBC jar
spark = SparkSession.builder \
    .appName("PostgreSQL Integration Example") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.0") \
    .config("spark.jars", rel_path) \
    .config("spark.executor.extraClassPath", rel_path) \
    .config("spark.driver.extraClassPath", rel_path) \
    .getOrCreate()

26/06/08 18:02:01 WARN Utils: Your hostname, codespaces-f5202d resolves to a loopback address: 127.0.0.1; using 10.0.0.231 instead (on interface eth0)
26/06/08 18:02:01 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/usr/local/sdkman/candidates/spark/3.5.1/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/vscode/.ivy2/cache
The jars for the packages stored in: /home/vscode/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-2a631bce-7988-4069-bde9-c27c3913755b;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.4.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.4.0 in central
	found org.apache.kafka#kafka-clients;3.3.2 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.9.1 in central
	found org.slf4j#slf4j-api;2.0.6 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
downloading https://repo1.maven.org/maven2/org/apache/spark/spa

### Step 2: Define Connection Properties

To connect to PostgreSQL, specify the connection properties including the URL, user credentials, and the JDBC driver:

In [6]:
# Connection URL and properties for PostgreSQL
url = "jdbc:postgresql://fhtw-big-data.postgres.database.azure.com/music_store"
postgres_options = {
    "url": url,
    "user": "student",
    "password": "reRZ2pjg1WxqlwjU",
    "driver": "org.postgresql.Driver"
}

## Enhancing Data Operations with JDBC and Spark DataFrames

### Reading Data from PostgreSQL Using JDBC in Apache Spark
When pulling data from a PostgreSQL database using JDBC with Spark, the process can be fine-tuned to suit specific needs, such as performance optimization and minimizing network overhead. You can perform direct table reads or use custom SQL queries to control the dataset size and structure before it is even loaded into Spark.

### Direct Table Reads
Direct table reads are straightforward; they involve loading the entire table into a DataFrame. This method is simple but might not be efficient if the table is large or if only a subset of the data is needed.

```python
# Loading an entire table
full_table_df = spark.read \
    .jdbc(url=url, table="full_table_name", properties=postgres_options)
full_table_df.show()

In [7]:
full_table_df = spark.read \
    .jdbc(url=url, table="tracks", properties=postgres_options)
full_table_df.show()

+---+--------------------+--------+-------------+--------+--------------------+------------+--------+----------+
| id|                name|album_id|media_type_id|genre_id|            composer|milliseconds|   bytes|unit_price|
+---+--------------------+--------+-------------+--------+--------------------+------------+--------+----------+
|  1|For Those About T...|       1|            1|       1|Angus Young, Malc...|      343719|11170334|      0.99|
|  2|   Balls to the Wall|       2|            2|       1|                NULL|      342562| 5510424|      0.99|
|  3|     Fast As a Shark|       3|            2|       1|F. Baltes, S. Kau...|      230619| 3990994|      0.99|
|  4|   Restless and Wild|       3|            2|       1|F. Baltes, R.A. S...|      252051| 4331779|      0.99|
|  5|Princess of the Dawn|       3|            2|       1|Deaffy & R.A. Smi...|      375418| 6290521|      0.99|
|  6|Put The Finger On...|       1|            1|       1|Angus Young, Malc...|      205662| 671

## Custom SQL Queries

Custom SQL queries allow more granular control over the data read operation. By defining a specific SQL query, you can filter and process data within the database before it reaches Spark, which can significantly enhance performance.

In [8]:
# Reading data using a custom SQL query
query = "(SELECT id, name FROM tracks WHERE composer = 'AC/DC') AS subquery"
filtered_df = spark.read.jdbc(url=url, table=query, properties=postgres_options)
filtered_df.show()

+---+--------------------+
| id|                name|
+---+--------------------+
| 15|             Go Down|
| 16|         Dog Eat Dog|
| 17|   Let There Be Rock|
| 18|      Bad Boy Boogie|
| 19|       Problem Child|
| 20|            Overdose|
| 21|Hell Ain't A Bad ...|
| 22|   Whole Lotta Rosie|
+---+--------------------+



### Managed Tables
When you save a DataFrame as a managed table, Spark manages both the metadata and the data of the table. If the table is deleted, both the metadata and the data are also deleted.

In [9]:
# convert the JSON into a table
df = spark.read.json('./data/music_log.json')
df.show(10)

+--------------------+---------+---------+------+-------------+---------+---------+-----+--------------------+------+--------+-----+-------------+---------+--------------------+------+-------------+--------------------+------+
|              artist|     auth|firstName|gender|itemInSession| lastName|   length|level|            location|method|    page|pagse| registration|sessionId|                song|status|           ts|           userAgent|userId|
+--------------------+---------+---------+------+-------------+---------+---------+-----+--------------------+------+--------+-----+-------------+---------+--------------------+------+-------------+--------------------+------+
|       Showaddywaddy|Logged In|  Kenneth|     M|          112| Matthews|232.93342| paid|Charlotte-Concord...|   PUT|NextSong| NULL|1509380319284|     5132|Christmas Tears W...|   200|1513720872284|"Mozilla/5.0 (Win...|  1046|
|          Lily Allen|Logged In|Elizabeth|     F|            7|    Chase|195.23873| free|Shr

## Using saveAsTable with DataFrames in Spark

The saveAsTable method in Spark is used to write a DataFrame as a table in the Spark SQL catalog. This can either create a new table or overwrite an existing one. It’s a key feature for managing data within Spark’s metastore and supports both managed and external tables.



### Considerations for Using saveAsTable

* Table Formats: You can specify various formats like Parquet, ORC for storing the table. Parquet is often used due to its efficiency in both storage and query performance within Spark.
* Save Modes: The mode option in saveAsTable can be set to “append”, “overwrite”, “ignore”, or “errorifexists”, providing flexibility depending on the operational requirements.


In [10]:
# Writing DataFrame as a managed table
spark.sql("DROP TABLE IF EXISTS managed_table_name")
df.write.format("parquet").saveAsTable("managed_table_name", mode="overwrite")

### Saving to an External Database (e.g., PostgreSQL)

When you wish to write a DataFrame to an external database like PostgreSQL, you would typically use the JDBC API. This involves specifying the format as "jdbc", providing a URL, and other connection properties:

In [12]:
# Writing DataFrame as an external table to db streaming_data
# the schema name is streaming_data and the table name is STUDENT_ID_json_logs

student_id = 'ds25m046' # Replace with your actual student ID
table_name = f"streaming_data.{student_id}_json_logs"

df.write.jdbc(url=url, table=table_name, mode="overwrite", properties=postgres_options)

26/06/08 18:04:26 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


### Example, analyze songs played by user gender and store into table

If you’re interested in the differences in song preferences between genders, you might group by both song title and user gender.

In [13]:
from pyspark.sql.functions import desc

# Group by song and gender, then count occurrences
songs_by_gender_df = df.groupBy("song", "gender").count().orderBy("song", desc("count"))

# Show results
songs_by_gender_df.show()    

+--------------------+------+-----+
|                song|gender|count|
+--------------------+------+-----+
|                NULL|     M|  762|
|                NULL|     F|  555|
|                NULL|  NULL|  336|
|                  #1|     F|    1|
|              $29.00|     F|    1|
|              & Down|     M|    1|
|           &And Ever|     M|    1|
|        ' Cello Song|     M|    1|
|  '97 Bonnie & Clyde|     F|    1|
|     'Round Midnight|     M|    1|
|'Round Midnight (...|     F|    1|
|    'Till I Collapse|     M|    4|
|    'Till I Collapse|     F|    2|
|(515) (Album Vers...|     F|    1|
|(F)lannigan's Bal...|     F|    1|
|(Hurricane) The F...|     M|    1|
|(I Love You) For ...|     M|    1|
|(I've Had) The Ti...|     M|    2|
|(If You're Wonder...|     M|    3|
|(Let Me Up) I've ...|     F|    1|
+--------------------+------+-----+
only showing top 20 rows



In [14]:
table_name = f"streaming_data.{student_id}_songs_by_gender"

songs_by_gender_df.write.jdbc(url=url, table=table_name, mode="overwrite", properties=postgres_options)

In [15]:
spark.stop()